## 1. Preparation

### 1.1 Import dependencies

In [1]:
import os
import json

from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer
import datasets

from peft import LoraConfig, get_peft_model, TaskType
import torch

In [2]:
import warnings
warnings.filterwarnings('ignore')

### 1.2 Set constants

In [3]:
MODEL_NAME = "./models/Qwen/Qwen3-4B-Instruct-2507" #local address of Qwen3-4B-Instruct
MAX_LENGTH = 4096 #tokens of one input usually between 900 - 2000
SEED = 42

torch.manual_seed(SEED)

### 1.2 Fix and merge the data
Merge all the jsonl to one. Some data entries have escape slash issue. Need to fix it.

In [4]:
# Read the file line by line, parse each, and re-save
input_folder = "./data/generation"
output_file = "./data/final/LoRA_samples_sum_fixed.jsonl"
merged = []

for filename in os.listdir(input_folder):
    if filename.endswith(".jsonl"):
        file_path = os.path.join(input_folder, filename)
        print(f"Start to process: {file_path}")

        with open(file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f, 1):
                try:
                    # Parse the JSON to validate it
                    data = json.loads(line.strip())
                    merged.append(data)
                except json.JSONDecodeError as e:
                    print(f"[ERROR] {filename} line {i}: {e}")
                    print(f"Content: {line[:200]}")

        print(f"Finished processing: {file_path}")

# Write back with proper encoding
with open(output_file, 'w', encoding='utf-8') as f:
    for data in merged:
        f.write(json.dumps(data, ensure_ascii=False) + '\n')

print(f"\nFinished. Total valid samples: {len(merged)}")

Start to process: ./data/LoRA_samples_intention.jsonl
[ERROR] LoRA_samples_intention.jsonl line 282: Invalid \escape: line 1 column 1635 (char 1634)
Content: {"messages":[{"role":"user","content":"## === 你的角色描述和背景信息（仅供参考） ===\n你是宜宾品牌汽车专场展会的客服，负责邀约当地意向客户进场参观并解答疑问。\n\n## === 你的核心任务（必须遵守） ===\n你需要根据以下指示判断**最后一次用户输入**的意图，然后**仅输出一个JSON对象**，绝对不能是其他任何格式\n\n### 重要
[ERROR] LoRA_samples_intention.jsonl line 286: Invalid \escape: line 1 column 1557 (char 1556)
Content: {"messages":[{"role":"user","content":"## === 你的角色描述和背景信息（仅供参考） ===\n你是南昌教育展的客服，需要和本地家长和学生沟通本地知名学校的展览\n\n## === 你的核心任务（必须遵守） ===\n你需要根据以下指示判断**最后一次用户输入**的意图，然后**仅输出一个JSON对象**，绝对不能是其他任何格式\n\n### 重要规则：\
Finished processing: ./data/LoRA_samples_intention.jsonl
Start to process: ./data/LoRA_samples_knowledge.jsonl
[ERROR] LoRA_samples_knowledge.jsonl line 324: Invalid \escape: line 1 column 421 (char 420)
Content: {"messages": [{"role": "user", "content": "## === 你的角色描述和背景信息（仅供参考） ===\n你是佛山陶瓷博览会的电话营销专员，负责邀请佛山的建材经销商和业主参加即将举办的陶瓷新品展。\n\n#

## 2. Dataset

In [5]:
# Load dataset
ds = load_dataset("json", data_files=output_file)["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
print(ds)

Dataset({
    features: ['messages'],
    num_rows: 1020
})


In [7]:
# Shuffle the datasets
shuffled_ds = ds.shuffle(seed=SEED)

In [8]:
shuffled_ds[:3]

{'messages': [[{'role': 'user',
    'content': "## === 你的角色描述和背景信息（仅供参考） ===\n你是洛阳家装节的客服，负责邀请本地业主参观展会。\n\n## === 你的核心任务（必须遵守） ===\n你需要根据以下指示判断**最后一次用户输入**的意图，然后**仅输出一个JSON对象**，绝对不能是其他任何格式\n\n### 重要规则：\n1. 你只负责判断意图，**不进行对话**\n2. 无论对话历史如何，你的**输出必须是且仅是一个JSON对象**，不得输出为空、不得输出其他格式的数据\n3. 禁止输出任何自然语言、解释或额外内容\n4. 如果之前的回复是自然语言，忽略它并继续输出JSON\n5. JSON必须包含且仅包含两个字段：`input_summary`、`intention_id`。字段值必须为字符串，格式必须严格遵循：{'input_summary': '...', 'intention_id': '...'}\n\n### JSON字段说明：\n1. **input_summary**\n- 务必参考**智能助手和用户的对话历史**，用不超过10个汉字，清晰简洁地概括**最后一次用户输入**的核心内容\n- 示例：“用户有兴趣参加活动”、“用户不想被打扰”、“用户询问你是谁”\n\n2. **intention_id**\n- 务必参考**智能助手和用户的对话历史**，判断**最后一次用户输入**的意图\n- 仅可从以下【意图库列表】或【知识库列表】中，根据**意图名称**和**意图说明**，选择**唯一最匹配**的意图\n- 若最后一次用户输入无法匹配任一意图，输出 `intention_id` 为 'others'\n- 如果最后一次用户输入的语义匹配某一条意图的**意图名称**和**意图说明**，则将这条意图的**意图id**输出为 `intention_id`\n- 禁止自行创建或修改 `intention_id`，其值必须严格来自两个列表中的意图id，或为 'others'\n- 选择优先级说明：\n  - 如果两个列表均有内容，首先使用【意图库列表】判别用户的意图，当匹配成功，将**意图id**输出为 `intention_id`；若无法匹配，才使用【知识库列表】判别用户的意

Under a dict, there is one key "messages", the value is a list. Each item in the list is a data sample (also a list).

### 2.1 Split the dataset
Split into train (75%), validation (15%), test (10%)

In [10]:
train_temp = shuffled_ds.train_test_split(test_size=0.25, seed=SEED)

val_test = train_temp['test'].train_test_split(test_size=0.4, seed=42)

ds = DatasetDict({
    'train': train_temp['train'],        # 75%
    'validation': val_test['train'],     # 15%
    'test': val_test['test']            # 10%
})

In [39]:
ds["train"]

Dataset({
    features: ['messages'],
    num_rows: 765
})

In [12]:
ds["validation"]

Dataset({
    features: ['messages'],
    num_rows: 153
})

In [13]:
ds["test"]

Dataset({
    features: ['messages'],
    num_rows: 102
})

### 2.2 Tokenization

In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side='right')

In [15]:
tokenizer

Qwen2Tokenizer(name_or_path='Qwen/Qwen3-4B-Instruct-2507', vocab_size=151643, model_max_length=1010000, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=

In [16]:
# The tokenizer already has PAD token and EOS token. There is no need for extra set up.
print(f"\\nTokenizer info:")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

\nTokenizer info:
PAD token: <|endoftext|> (ID: 151643)
EOS token: <|im_end|> (ID: 151645)


Process function

In [17]:
def process_func(example):
    """
    Tokenize with Qwen chat template
    Mask instruction tokens so model only learns from assistant response
    """
    messages = example["messages"]
    
    instruction_text = tokenizer.apply_chat_template(
        [messages[0]], # can only take a list as argument
        tokenize=False,
        add_generation_prompt=True # add token of '<|im_start|>assistant'
    )

    instruction_tokens = tokenizer(
        instruction_text,
        add_special_tokens=False
    )

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False # This is the full sample, no need to add anything extra
    )

    full_tokens = tokenizer(
        full_text+tokenizer.eos_token,
        truncation=True,
        max_length=MAX_LENGTH,
        add_speicial_tokens=False
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    instruction_len = len(instruction_tokens["input_ids"])
    labels = [-100] * instruction_len + input_ids[instruction_len:]

    return{
        "input_ids" : input_ids,
        "attention_mask" : attention_mask,
        "labels" : labels
    }

In [19]:
tokenized_ds = ds.map(process_func,
                      remove_columns=ds["train"].column_names,
                      num_proc=4,
                      desc="Tokenizing"
                     )

Tokenizing (num_proc=4):   0%|          | 0/765 [00:00<?, ? examples/s]

Tokenizing (num_proc=4):   0%|          | 0/153 [00:00<?, ? examples/s]

Tokenizing (num_proc=4):   0%|          | 0/102 [00:00<?, ? examples/s]

In [20]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 765
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 153
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 102
    })
})

In [30]:
len(tokenized_ds["train"]["input_ids"][5])

1387

In [32]:
# Check sequence lengths
max_len = max(len(x) for x in tokenized_ds["train"]["input_ids"])
avg_len = sum(len(x) for x in tokenized_ds["train"]["input_ids"]) / len(tokenized_ds["train"])
print(f"Sequence length - Max: {max_len}, Avg: {avg_len:.1f}")

Sequence length - Max: 2043, Avg: 1474.7


In [27]:
tokenized_ds["train"][2]

{'input_ids': [151644,
  872,
  198,
  565,
  2049,
  220,
  103929,
  100780,
  53481,
  33108,
  102193,
  27369,
  9909,
  110079,
  7552,
  2049,
  198,
  105043,
  105842,
  105530,
  76313,
  91453,
  36993,
  9370,
  100257,
  94237,
  105041,
  3837,
  107050,
  75108,
  26939,
  40542,
  8903,
  104743,
  59532,
  100410,
  22697,
  100170,
  99488,
  100755,
  101951,
  105530,
  23031,
  100052,
  71134,
  16628,
  111144,
  36993,
  3837,
  102333,
  100647,
  104541,
  3837,
  85106,
  87805,
  104331,
  104603,
  101973,
  101061,
  3407,
  565,
  2049,
  220,
  103929,
  100185,
  88802,
  9909,
  100645,
  105389,
  7552,
  2049,
  198,
  112735,
  100345,
  87752,
  102857,
  104317,
  334,
  117241,
  20002,
  31196,
  334,
  9370,
  111450,
  3837,
  101889,
  334,
  99373,
  66017,
  46944,
  5370,
  64429,
  334,
  3837,
  101244,
  53153,
  20412,
  92894,
  99885,
  68805,
  271,
  14374,
  220,
  99335,
  104190,
  28311,
  16,
  13,
  220,
  56568,
  91680,
  1

In [28]:
tokenizer.decode(tokenized_ds["train"][2])

'<|im_start|>user\n## === 你的角色描述和背景信息（仅供参考） ===\n你是县城家电展销会的邀约客服，本周五到周日将在利川市体育中心举办大型家电以旧换新直销会，厂家现场补贴，需要电话邀请本地居民参加。\n\n## === 你的核心任务（必须遵守） ===\n你需要根据以下指示判断**最后一次用户输入**的意图，然后**仅输出一个JSON对象**，绝对不能是其他任何格式\n\n### 重要规则：\n1. 你只负责判断意图，**不进行对话**\n2. 无论对话历史如何，你的**输出必须是且仅是一个JSON对象**，不得输出为空、不得输出其他格式的数据\n3. 禁止输出任何自然语言、解释或额外内容\n4. 如果之前的回复是自然语言，忽略它并继续输出JSON\n5. JSON必须包含且仅包含两个字段：`input_summary`、`intention_id`。字段值必须为字符串，格式必须严格遵循：{\'input_summary\': \'...\', \'intention_id\': \'...\'}\n\n### JSON字段说明：\n1. **input_summary**\n- 务必参考**智能助手和用户的对话历史**，用不超过10个汉字，清晰简洁地概括**最后一次用户输入**的核心内容\n- 示例：“用户有兴趣参加活动”、“用户不想被打扰”、“用户询问你是谁”\n\n2. **intention_id**\n- 务必参考**智能助手和用户的对话历史**，判断**最后一次用户输入**的意图\n- 仅可从以下【意图库列表】或【知识库列表】中，根据**意图名称**和**意图说明**，选择**唯一最匹配**的意图\n- 若最后一次用户输入无法匹配任一意图，输出 `intention_id` 为 \'others\'\n- 如果最后一次用户输入的语义匹配某一条意图的**意图名称**和**意图说明**，则将这条意图的**意图id**输出为 `intention_id`\n- 禁止自行创建或修改 `intention_id`，其值必须严格来自两个列表中的意图id，或为 \'others\'\n- 选择优先级说明：\n  - 如果两个列表均有内容，则同时使用【知识库列表】和【意图库列表】判别用户的意图，不分先后，当匹配成功，将**意图id**输出为 

In [78]:
tokenizer.decode(list(filter(lambda x:x!=-100,tokenized_ds["train"][2]["labels"])))

'{"input_summary": "用户询问地点", "intention_id": "q7r8s9t0u1v2w3x4"}<|im_end|>'

In [79]:
tokenized_ds["train"][2]["input_ids"]

[33975,
 25,
 7704,
 2049,
 220,
 103929,
 100780,
 53481,
 33108,
 102193,
 27369,
 9909,
 110079,
 7552,
 2049,
 198,
 105043,
 114440,
 45629,
 102941,
 9370,
 105041,
 3837,
 100668,
 72064,
 36667,
 104235,
 9370,
 20002,
 3837,
 81167,
 41146,
 64471,
 113205,
 90395,
 106185,
 101888,
 105674,
 106730,
 86119,
 3407,
 565,
 2049,
 220,
 103929,
 100185,
 88802,
 9909,
 100645,
 105389,
 7552,
 2049,
 198,
 112735,
 100345,
 87752,
 102857,
 104317,
 334,
 117241,
 20002,
 31196,
 334,
 9370,
 111450,
 3837,
 101889,
 334,
 99373,
 66017,
 46944,
 5370,
 64429,
 334,
 3837,
 101244,
 53153,
 20412,
 92894,
 99885,
 68805,
 271,
 14374,
 220,
 99335,
 104190,
 28311,
 16,
 13,
 220,
 56568,
 91680,
 100668,
 104317,
 111450,
 3837,
 334,
 16530,
 71817,
 105051,
 1019,
 17,
 13,
 220,
 100783,
 105051,
 100022,
 100007,
 3837,
 103929,
 334,
 66017,
 100645,
 20412,
 100136,
 99373,
 101909,
 5370,
 64429,
 334,
 3837,
 99925,
 66017,
 50647,
 5373,
 99925,
 66017,
 92894,
 68805,

In [80]:
tokenized_ds["train"][2]["labels"]

[-100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,

In [85]:
len(tokenized_ds["train"][2]["labels"])

1523

## 3. Create model

In [34]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
    # low_cpu_mem_usage = True # depreaciated, auto enbaled by default
)

In [22]:
model.device

device(type='cpu')

In [ ]:
# Disable cache during training
model.config.use_cache = False

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()

In [23]:
model

BloomForCausalLM(
  (transformer): BloomModel(
    (word_embeddings): Embedding(46145, 2048)
    (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
    (h): ModuleList(
      (0-23): 24 x BloomBlock(
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (self_attention): BloomAttention(
          (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
          (dense): Linear(in_features=2048, out_features=2048, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (mlp): BloomMLP(
          (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
          (gelu_impl): BloomGelu()
          (dense_4h_to_h): Linear(in_features=8192, out_features=2048, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
  )
  (l

In [24]:
sum(param.numel() for param in model.parameters())

1303111680

## 4. LoRA Configuration

### 4.1 PEFT step 1, configuration

In [35]:
config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r = 32,
    lora_alpha = 64,
    lora_dropout = 0.05,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none"
)

In [36]:
config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=32, target_modules={'gate_proj', 'k_proj', 'q_proj', 'down_proj', 'up_proj', 'o_proj', 'v_proj'}, exclude_modules=None, lora_alpha=64, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)

### 4.2 PEFT step 2, create model

In [30]:
model = get_peft_model(model, config)
model = torch.compile(model)

In [31]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(46145, 2048)
        (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0-23): 24 x BloomBlock(
            (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=6144, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=6144, bias=False)
                )
                (lora_embedding_A): Paramete

In [32]:
model.print_trainable_parameters()

trainable params: 1,572,864 || all params: 1,304,684,544 || trainable%: 0.1206


The parameters that need to be adjusts is significantly decreased from 1.3B to 1.5M.

In [33]:
for name, parameter in model.named_parameters():
    print(name)

base_model.model.transformer.word_embeddings.weight
base_model.model.transformer.word_embeddings_layernorm.weight
base_model.model.transformer.word_embeddings_layernorm.bias
base_model.model.transformer.h.0.input_layernorm.weight
base_model.model.transformer.h.0.input_layernorm.bias
base_model.model.transformer.h.0.self_attention.query_key_value.base_layer.weight
base_model.model.transformer.h.0.self_attention.query_key_value.base_layer.bias
base_model.model.transformer.h.0.self_attention.query_key_value.lora_A.default.weight
base_model.model.transformer.h.0.self_attention.query_key_value.lora_B.default.weight
base_model.model.transformer.h.0.self_attention.dense.weight
base_model.model.transformer.h.0.self_attention.dense.bias
base_model.model.transformer.h.0.post_attention_layernorm.weight
base_model.model.transformer.h.0.post_attention_layernorm.bias
base_model.model.transformer.h.0.mlp.dense_h_to_4h.weight
base_model.model.transformer.h.0.mlp.dense_h_to_4h.bias
base_model.model.tra

## 5. Configure the training

In [34]:
args = TrainingArguments(
    output_dir = "./fine_tuning_output", #to store the prediction results and checkpoints of the model file
    per_device_train_batch_size = 4, #8 by default
    per_device_eval_batch_size = 4,
    gradient_accumulation_steps = 4, # Effective batch size = 16

    num_train_epochs=3,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    # ~800 training samples
    warmup_ratio=0.1,
    weight_decay=0.02, # for overfitting prevention

    # ~8000 training samples
    # warmup_ratio=0.03,
    # weight_decay=0.01
    
    max_grad_norm=1.0,
    
    # Logging
    logging_steps=10,
    logging_first_step=True,

    # Evaluation
    evaluation_strategy="steps",
    eval_steps=50, # ~800 training samples
    # eval_steps=200, #~8000 training samples

    # Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Precision
    bf16=True,
    fp16=False,
    
    # Memory optimization
    gradient_checkpointing=True,
    
    # Performance
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    
    # Reproducibility
    seed=42,
    data_seed=42,
    
    # Reporting
    report_to="none",
    
    # Other
    remove_unused_columns=False
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8  # Optimize for tensor cores
)

## 6. Create the trainer and train

In [35]:
trainer = Trainer(
    model = model, #the model with frozen parameters
    args = args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator = data_collator   
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [37]:
trainer.train()

## 7. Save model

In [38]:
output_dir = "./lora_output"

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("Model saved successfully!")

device(type='mps', index=0)

## 8. Evaluation on test set

#### Helper functions

In [41]:
def extract_intention_id(response_text):
    """
    Extract intention_id from JSON response
    Handles various formats and errors
    """
    try:
        # Try to parse as JSON
        data = json.loads(response_text.strip())
        return data.get("intention_id", None)
    except json.JSONDecodeError:
        # Try regex extraction if JSON parsing fails
        match = re.search(r'"intention_id"\\s*:\\s*"([^"]+)"', response_text)
        if match:
            return match.group(1)
        return None


In [ ]:
def evaluate_intention_accuracy(model, tokenizer, test_data, name, batch_size=8):
    """
    Evaluate model accuracy on intention_id prediction
    """
    print(f"\\nEvaluating {name} on intention_id...")
    
    model.eval()
    correct = 0
    total = 0
    invalid_responses = 0
    
    for i in tqdm(range(0, len(test_data), batch_size), desc="Evaluating"):
        batch = test_data[i:i+batch_size]
        
        prompts = []
        ground_truth_ids = []
        
        for example in batch:
            messages = example["messages"]
            
            # Extract ground truth intention_id
            try:
                ground_truth_json = json.loads(messages[1]["content"])
                ground_truth_ids.append(ground_truth_json["intention_id"])
            except:
                ground_truth_ids.append(None)
                continue
            
            # Prepare prompt
            prompt = tokenizer.apply_chat_template(
                [messages[0]],
                tokenize=False,
                add_generation_prompt=True
            )
            prompts.append(prompt)
        
        # Tokenize batch
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # Evaluate predictions
        for j, output in enumerate(outputs):
            if ground_truth_ids[j] is None:
                continue
                
            prediction_text = tokenizer.decode(
                output[inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            ).strip()
            
            # Extract intention_id from prediction
            predicted_id = extract_intention_id(prediction_text)
            
            if predicted_id is None:
                invalid_responses += 1
            elif predicted_id == ground_truth_ids[j]:
                correct += 1
            
            total += 1
    
    accuracy = (correct / total) * 100 if total > 0 else 0
    valid_rate = ((total - invalid_responses) / total) * 100 if total > 0 else 0
    
    print(f"\\n{name} Results:")
    print(f"  Correct predictions:  {correct}/{total} ({accuracy:.2f}%)")
    print(f"  Valid JSON responses: {total - invalid_responses}/{total} ({valid_rate:.2f}%)")
    print(f"  Invalid responses:    {invalid_responses}")
    
    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "invalid": invalid_responses,
        "valid_rate": valid_rate
    }

### 8.1 Evaluate the LoRA model

In [ ]:
lora_metrics = trainer.evaluate(tokenized_ds["test"])
lora_loss = lora_metrics["eval_loss"]
lora_perplexity = torch.exp(torch.tensor(lora_loss)).item()

lora_results = evaluate_intention_accuracy(
    trainer.model,
    tokenizer,
    dataset["test"],
    "Fine-tuned LoRA",
    batch_size=8
)

### 8.2 Evaluate the base model

In [ ]:
# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

In [ ]:
base_trainer = Trainer(
    model=base_model,
    args=training_args,
    eval_dataset=tokenized_ds["test"],
    data_collator=data_collator
)

In [ ]:
base_metrics = base_trainer.evaluate()
base_loss = base_metrics["eval_loss"]
base_perplexity = torch.exp(torch.tensor(base_loss)).item()
# Evaluate base model accuracy
base_results = evaluate_intention_accuracy(
    base_model,
    tokenizer,
    dataset["test"],
    "Base Qwen3-4B",
    batch_size=8
)

In [ ]:
del base_model
del base_trainer
torch.cuda.empty_cache()

### 8.3 Final comparison

In [ ]:
print("FINAL RESULTS COMPARISON")
print("="*60)

# Loss comparison
print("\\n📊 Loss Metrics:")
print(f"  Base Model:      {base_loss:.4f} (perplexity: {base_perplexity:.2f})")
print(f"  Fine-tuned:      {lora_loss:.4f} (perplexity: {lora_perplexity:.2f})")
loss_improvement = base_loss - lora_loss
print(f"  Improvement:     {loss_improvement:.4f} ({loss_improvement/base_loss*100:+.2f}%)")

# Accuracy comparison
print("\\n🎯 Intention ID Accuracy:")
print(f"  Base Model:      {base_results['accuracy']:.2f}% ({base_results['correct']}/{base_results['total']})")
print(f"  Fine-tuned:      {finetuned_results['accuracy']:.2f}% ({finetuned_results['correct']}/{finetuned_results['total']})")
accuracy_improvement = finetuned_results['accuracy'] - base_results['accuracy']
print(f"  Improvement:     {accuracy_improvement:+.2f}%")

# Valid JSON rate comparison
print("\\n✅ Valid JSON Response Rate:")
print(f"  Base Model:      {base_results['valid_rate']:.2f}%")
print(f"  Fine-tuned:      {finetuned_results['valid_rate']:.2f}%")
valid_improvement = finetuned_results['valid_rate'] - base_results['valid_rate']
print(f"  Improvement:     {valid_improvement:+.2f}%")

In [ ]:
# Final verdict
if accuracy_improvement > 0 and loss_improvement > 0:
    print(f"✅ SUCCESS! Fine-tuning improved both accuracy ({accuracy_improvement:+.2f}%) and loss ({loss_improvement:.4f})")
elif accuracy_improvement > 0:
    print(f"✅ Fine-tuning improved accuracy by {accuracy_improvement:+.2f}%")
elif loss_improvement > 0:
    print(f"⚠️  Loss improved but accuracy decreased by {accuracy_improvement:.2f}%")
else:
    print(f"❌ Fine-tuning did not improve the model")

print("="*60)
print(f"\\n💾 Fine-tuned model saved to: {OUTPUT_DIR}")